In [1]:
import os
import datetime
import instructor
import ollama
import stanza
from pydantic import BaseModel, Field

In [2]:
import sys
import torch
import numpy as np
from importlib import reload
from pathlib import Path

import re
import json as _json

# Intercept and patch the global torch load function
original_load = torch.load
def patched_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return original_load(*args, **kwargs)
torch.load = patched_load

# Reload stanza if it has already been initialized in this session
if 'stanza' in sys.modules:
    reload(sys.modules['stanza'])

# Tell PyTorch's unpickler that NumPy's multiarray reconstructor is safe to load
torch.serialization.add_safe_globals([
    np.core.multiarray._reconstruct,
    np.dtype,
])

import spacy_stanza
import random
stanza.download('he')

import json
from jsonschema import validate, ValidationError

In [3]:
# Story inputs — edit only story_brief for a new episode
story_brief = {
    "level": "CEFR A1/A2",
    "character_name": "{{pet_name}}",
    "allowed_pet_type_ids": [10],
    "quest_line_id": "garden_adventure",
    "episode_number": 1,
    "episode_title": "The Empty Basket",
    "story_overview": """
        Faun is very hungry for a sweet, crunchy carrot.
        He decides to spend the day gardening in the sunny garden.
        He takes a basket of tools, plants a seed, waters it, and proudly harvests his first carrot.
    """,
    "vocabulary_targets" = {
        "hunger": "רעב",
        "basket": "סל",
        "garden": "גן",
        "seed": "זרע",
        "carrot": "גזר"
    }
}

level = story_brief["level"]
character_name = story_brief["character_name"]
allowed_pet_type_ids = story_brief["allowed_pet_type_ids"]
quest_line_id = story_brief["quest_line_id"]
episode_number = story_brief["episode_number"]
episode_title = story_brief["episode_title"]
story_overview = story_brief["story_overview"]
word_list = story_brief["vocabulary_targets"]

# Derived values used by the rest of the notebook
story_title = episode_title
formatted_words = "\n".join([f"- {word}" for word in word_list])
quest_id = f"{quest_line_id}_ep{episode_number:02d}"
animal_folder = quest_line_id

# English generation prompt (returns JSON array of sentence objects)
large_prompt_en = f"""
You are a JSON API that outputs ONLY valid JSON.

Write episode {episode_number:02d} of the quest line "{quest_line_id}" as a children's story JSON array.

Episode title:
{episode_title}

Main character:
{character_name}

Story overview:
{story_overview}

Required vocabulary words:
{formatted_words}

Rules:
- Output ONLY JSON.
- No markdown.
- No explanations.
- No prose outside JSON.
- No comments.
- No trailing commas.
- Each sentence must be short.
- One action per sentence.
- Use simple CEFR A1/A2 English.
- Generate 5 to 8 sentences.
- Use the required vocabulary words naturally across the episode.

Required format:

[
  {{
    "id": "s1",
    "male": "Faun was hungry.",
    "female": "Faun was hungry."
  }},
  {{
    "id": "s2",
    "male": "He walked to the garden.",
    "female": "He walked to the garden."
  }}
]
"""

# Generate timestamped filenames to prevent overrides
timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
filename = f"{timestamp}_{quest_id}"

In [4]:
# Per-sentence orchestration: split generation and translation into two functions
def make_translate_prompt(sentence_text, character_name):
    return f"""
    Translate to Hebrew.
    
    English:
    {sentence_text}
    
    Rules:
    - No explanations
    - No notes
    - No list
    - No markdown
    - No English
    - No quotes
    - Hebrew only
    
    Output:
    """

In [5]:
def feminize_english_sentence(text, character_name):

    replacements = {
        " he ": " she ",
        " He ": " She ",
        " him ": " her ",
        " Him ": " Her ",
        " his ": " her ",
        " His ": " Her ",
        " himself ": " herself ",
        " Himself ": " Herself ",
        " boy ": " girl ",
        " Boy ": " Girl ",
    }

    result = f" {text} "

    for old, new in replacements.items():
        result = result.replace(old, new)

    return result.strip()

In [6]:
def call_ai(prompt, temp=0.2, model_name='llama3.1'):
    """Call the model and return a best-effort text string.
        @temp - higher temp is higher creativity
    """
    response = ollama.generate(
        model=model_name, 
        prompt=prompt,
        options={
            "temperature": temp,
            "stop": [
                "\n\n",
                "Explanation:",
                "שלב אחר שלב:",
                "1."
            ]
        }
    )
    try:
        if isinstance(response, dict):
            if 'response' in response:
                return response['response']
            if 'generations' in response:
                gens = response['generations']
                if isinstance(gens, list) and gens:
                    first = gens[0]
                    if isinstance(first, dict):
                        return first.get('male', str(first))
                    return str(first)
            if 'choices' in response:
                choices = response['choices']
                if isinstance(choices, list):
                    return '\n'.join([c.get('male', '') if isinstance(c, dict) else str(c) for c in choices])
        return response['response']
    except Exception:
        return response['response']

def generate_english_sentences(story_overview, character_name, animal_folder, filename, model_name='llama3.1'):
    # Generate English sentences JSON (single file), with robust parsing and normalized sequential ids
    prompt = large_prompt_en
    raw = call_ai(prompt, temp=0.2, model_name=model_name)

    raw_text = raw.strip() if raw else ''

    english_sentences = None

    try:
        english_sentences = _json.loads(raw_text)

    except Exception:

        try:
            match = re.search(
                r'(\[\s*\{.*?\}\s*\])',
                raw_text,
                re.DOTALL
            )

            if match:
                extracted = match.group(1)

                english_sentences = _json.loads(extracted)

        except Exception as e:
            print("JSON extraction failed")
            print(e)

    if not english_sentences:
        raise ValueError(
            "Failed to parse valid sentence JSON from model output"
        )
    
    # Normalize ids so they are contiguous: s1, s2, s3...
    for i, s in enumerate(english_sentences):
        s['id'] = f's{i+1}'
        if 'male' not in s:
            s['male'] = ''
        if 'female' not in s:
            s['female'] = ''

    # Persist a single English JSON file
    try:
        dir_name = f'./quests/{animal_folder}'
        if dir_name and not os.path.exists(dir_name):
            os.makedirs(dir_name)
        with open(f'{dir_name}/{filename}_en.json', 'w', encoding='utf-8') as f:
            f.write(_json.dumps(english_sentences, ensure_ascii=False, indent=2))
        # Save readable markdown version
        markdown_story = "\n\n".join([
            s['male'] for s in english_sentences
        ])

        with open(f'{dir_name}/{filename}_en.md', 'w', encoding='utf-8') as f:
            f.write(markdown_story)            
            
    except Exception:
        pass

    return english_sentences

def translate_sentences_to_he(english_sentences, character_name, model_name='aminadaven/dictalm2.0-instruct:q4_k_m'):
    """Translate each English sentence independently into male/female/neutral Hebrew variants."""
    story_sentences = []
    male_lines = []
    female_lines = []

    for s in english_sentences:
        sid = s.get('id')
        en_male = s.get('male', '')
        en_female = s.get('female', '')
        
        male = call_ai(make_translate_prompt(en_male, character_name), temp=0.1, model_name=model_name)
        female = call_ai(make_translate_prompt(en_female, character_name), temp=0.1, model_name=model_name)

        male = clean_hebrew_output(male)
        female = clean_hebrew_output(female)
        
        story_sentences.append({'id': sid, 'male': male.strip(), 'female': female.strip(), 'neutral': male.strip()})
        male_lines.append(male.strip())
        female_lines.append(female.strip())

    story_text_he = {
        'male': '\n'.join(male_lines),
        'female': '\n'.join(female_lines),
        'neutral': '\n'.join(male_lines),
    }
    return story_sentences, story_text_he



In [7]:
def clean_hebrew_output(text):

    if not text:
        return ""

    text = text.strip()

    # Remove markdown/code fences
    text = re.sub(r'```.*?```', '', text, flags=re.DOTALL)

    # Remove common reasoning patterns
    blacklist_patterns = [
        r'שלב אחר שלב:.*',
        r'Rules:.*',
        r'Output:.*',
        r'תרגום:.*',
        r'Explanation:.*',
        r'1\..*',
        r'2\..*',
        r'3\..*',
        r'4\..*',
        r'5\..*',
    ]

    for pattern in blacklist_patterns:
        text = re.sub(
            pattern,
            '',
            text,
            flags=re.DOTALL
        )

    # Remove English-heavy lines
    cleaned_lines = []

    for line in text.splitlines():

        hebrew_chars = len(re.findall(r'[\u0590-\u05FF]', line))
        english_chars = len(re.findall(r'[A-Za-z]', line))

        # Keep lines that are mostly Hebrew
        if hebrew_chars >= english_chars:
            cleaned_lines.append(line)

    text = '\n'.join(cleaned_lines)

    # Collapse whitespace
    text = re.sub(r'\n+', '\n', text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [8]:
# Run orchestration
english_sentences = generate_english_sentences(story_overview, character_name, animal_folder, filename)
story_sentences, story_text_he = translate_sentences_to_he(english_sentences, character_name, "llama3.1")
print(f'Generated {len(english_sentences)} English sentences and {len(story_sentences)} Hebrew sentence variants')

## Generate Quiz

In [9]:
# 2. Initialize the Stanza Hebrew pipeline wrapper
# Use a safe load with a fallback to stanza.Pipeline to avoid recursion issues
try:
    nlp = spacy_stanza.load_pipeline("he")
except RecursionError as e:
    print('RecursionError when loading spacy_stanza pipeline:', e)
    import stanza as _stanza
    try:
        _stanza.download('he')
    except Exception:
        pass
    # Fallback: construct a stanza Pipeline directly (common processors for tokenization/pos/lemma)
    nlp = _stanza.Pipeline(lang='he', processors='tokenize,mwt,pos,lemma')
except Exception as e:
    # If other errors occur, re-raise so the notebook shows the failure clearly
    raise


In [10]:
def extract_cloze_quizzes(story_hebrew, target_words, story_sentences):
    """
    Create cloze quizzes anchored to story_sentences with sentence_id links.
    """

    docs = {
        "male": nlp(story_hebrew["male"]),
        "female": nlp(story_hebrew["female"]),
        "neutral": nlp(story_hebrew["neutral"])
    }

    cloze_quizzes = []

    # Build shared POS bank from all variants.
    pos_bank = {}
    for doc in docs.values():
        for token in doc:
            if token.is_alpha and not token.is_stop:
                pos_bank.setdefault(token.pos_, set()).add(token.lemma_)

    for sentence_variants in story_sentences:
        neutral_sentence = sentence_variants.get("neutral", "")
        neutral_doc = nlp(neutral_sentence)
        sentence_id = sentence_variants.get("id")

        if not sentence_id:
            continue

        for target in target_words:
            target_token = next(
                (
                    t for t in neutral_doc
                    if t.text == target or t.lemma_ == target
                ),
                None
            )

            if not target_token:
                continue

            prompts = {}

            for gender in ("male", "female", "neutral"):
                gender_sentence = sentence_variants.get(gender, sentence_variants.get("neutral", ""))
                gender_doc = nlp(gender_sentence)

                matching_token = next(
                    (
                        t for t in gender_doc
                        if t.lemma_ == target_token.lemma_ or t.text == target_token.text
                    ),
                    None
                )

                if matching_token:
                    blank_text = gender_sentence.replace(matching_token.text, "_____", 1)
                else:
                    blank_text = gender_sentence

                prompts[gender] = blank_text

            target_pos = target_token.pos_
            distractor_pool = list(pos_bank.get(target_pos, set()))
            distractor_pool = [
                d for d in distractor_pool
                if d != target_token.text and d != target_token.lemma_ and d != target
            ]

            fallbacks = ["בית", "חבר", "מקום", "ילד"]
            while len(distractor_pool) < 3:
                distractor_pool.append(random.choice(fallbacks))

            distractors = random.sample(distractor_pool, 3)
            options = distractors + [target_token.text]
            random.shuffle(options)

            cloze_quizzes.append({
                "type": "cloze",
                "sentence_id": sentence_id,
                "prompt": prompts,
                "answer": target_token.text,
                "options": options
            })

            break

    return cloze_quizzes

In [11]:
# 1. Define the distantlife quest schema to catch errors before deployment
QUEST_SCHEMA = {
    "type": "object",
    "properties": {
        "meta": {
            "type": "object",
            "properties": {
                "schema_version": {"type": "string", "const": "2.0.0"},
                "generator": {"type": "string", "const": "quest_pipeline_v1"},
                "generated_at": {"type": "string"}
            },
            "required": ["schema_version", "generator", "generated_at"]
        },
        "quest_id": {"type": "string"},
        "locale": {"type": "string"},
        "theme": {"type": "string"},
        "quest_type": {"type": "string"},
        "quest_line_id": {"type": "string"},
        "allowed_pet_type_ids": {
            "type": "array",
            "items": {"type": "integer"}
        },
        "unlock_cost": {"type": "integer"},
        "title": {"type": "string"},
        "summary": {"type": "string"},
        "review_status": {"type": "string"},
        "episodes": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "episode_id": {"type": "string"},
                    "title": {"type": "string"},
                    "story_sentences": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "id": {"type": "string"},
                                "male": {"type": "string"},
                                "female": {"type": "string"},
                                "neutral": {"type": "string"}
                            },
                            "required": ["id", "male", "female", "neutral"]
                        }
                    },
                    "quiz": {
                        "type": "object",
                        "properties": {
                            "questions": {
                                "type": "array",
                                "items": {
                                    "type": "object",
                                    "properties": {
                                        "type": {"type": "string"},
                                        "sentence_id": {"type": "string"},
                                        "answer": {"type": "string"},
                                        "options": {"type": "array"}
                                    },
                                    "required": ["type"]
                                }
                            }
                        },
                        "required": ["questions"]
                    }
                },
                "required": ["episode_id", "title", "story_sentences", "quiz"]
            }
        }
    },
    "required": [
        "meta",
        "quest_id",
        "locale",
        "theme",
        "quest_type",
        "quest_line_id",
        "allowed_pet_type_ids",
        "episodes"
    ]
}

In [12]:
# Raw inputs from your Ollama story generation step
# story_hebrew_raw = story_text_he if 'story_text_he' in globals() else (story_text_he if 'story_text_he' in globals() else None)
# story_english_raw = english_story_output if 'english_story_output' in globals() else None
vocab_targets = word_list

# Build aligned sentence structure and quizzes — prefer `story_sentences` produced by translation
if 'story_sentences' not in globals() or story_sentences is None:
    if 'story_text_he' in globals() and story_text_he:
        story_sentences = extract_story_sentences(story_text_he)
    else:
        story_sentences = []

quizzes = extract_cloze_quizzes(story_text_he if 'story_text_he' in globals() else {'male':'','female':'','neutral':''}, vocab_targets, story_sentences)

In [15]:
# Package into a combined quest + per-episode files (best-practice layout)

# Configuration: adjust if you prefer another structure
# Example output layout:
#  - quests/{quest_line}/{quest_id}.json          (combined quest with all episodes & locales)
#  - quests/{quest_line}/{quest_id}/episodes/{episode_id}.json  (per-episode files with locales)
#
# Use quest_id convention like "garden_adventure_ep01" so the episode number is part of the output name.

quest_line = quest_line_id
quest_cost = 50

# Ensure these variables exist in the notebook:
# - english_sentences: list of {'id','male','female'} (from generate_english_sentences)
# - story_sentences: list of {'id','male','female','neutral'} (from translate_sentences_to_he)
# - quizzes: list of quiz dicts (from extract_cloze_quizzes)
# - vocab_targets: list of target words

# Normalize English sentences to include 'neutral'
en_story_sentences = []
for s in english_sentences:
    en_story_sentences.append({
        "id": s.get("id"),
        "male": s.get("male", ""),
        "female": s.get("female", ""),
        "neutral": s.get("male", "")
    })

# Build locale episode objects (a single episode in this notebook run)
episode_id = quest_id
episode_en = {
    "episode_id": episode_id,
    "title": episode_title,
    "story_sentences": en_story_sentences,
    "quiz": {"questions": []},
    "vocabulary_targets": vocab_targets
}
episode_he = {
    "episode_id": episode_id,
    "title": episode_title,
    "story_sentences": story_sentences,
    "quiz": {"questions": quizzes},
    "vocabulary_targets": vocab_targets
}

# Combined quest object (all locales)
combined_quest = {
    "meta": {
        "schema_version": "2.0.0",
        "generator": "quest_pipeline_v1",
        "generated_at": datetime.datetime.now(datetime.UTC).isoformat()
    },
    "quest_id": quest_id,
    "quest_line_id": quest_line,
    "unlock_cost": quest_cost,
    "allowed_pet_type_ids": allowed_pet_type_ids,
    "locales": {
        "en": {
            "locale": "en",
            "theme": "vegetables",
            "title": episode_title,
            "summary": story_overview,
            "episodes": [episode_en]
        },
        "he": {
            "locale": "he",
            "theme": "vegetables",
            "title": episode_title,
            "summary": story_overview,
            "episodes": [episode_he]
        }
    }
}


In [14]:

# Paths
base_dir = Path("../quests") / quest_line / quest_id
combined_path = Path("../quests") / quest_line / f"{quest_id}.json"
episodes_dir = base_dir / "episodes"

# Write combined quest file
os.makedirs(combined_path.parent, exist_ok=True)
with open(combined_path, "w", encoding="utf-8") as f:
    json.dump(combined_quest, f, ensure_ascii=False, indent=2)

# Write per-episode files (one file per episode, containing localized episode data)
os.makedirs(episodes_dir, exist_ok=True)
# For each episode in combined_quest, build an episode-scoped object with locales
for ep in combined_quest["locales"]["en"]["episodes"]:
    ep_id = ep["episode_id"]
    per_episode_obj = {
        "meta": combined_quest["meta"],
        "quest_id": combined_quest["quest_id"],
        "quest_line_id": combined_quest.get("quest_line_id"),
        "episode_id": ep_id,
        "locales": {}
    }
    # attach each locale's episode by matching episode_id in locale entries
    for locale_code, locale_data in combined_quest["locales"].items():
        # find episode with same id in this locale
        match = next((e for e in locale_data.get("episodes", []) if e.get("episode_id") == ep_id), None)
        per_episode_obj["locales"][locale_code] = match or {}

    try:
        episode_path = episodes_dir / f"{ep_id}.json"
        with open(episode_path, "w", encoding="utf-8") as ef:
            json.dump(per_episode_obj, ef, ensure_ascii=False, indent=2)
    except ValidationError as e:
        print(f"Schema violation detected: {e.message}")

print(f"Written combined quest: {combined_path}")
print(f"Wrote {len(list(episodes_dir.glob('*.json')))} episode files to {episodes_dir}")